In [2]:
pip install numpy torch torchvision opencv-python matplotlib scikit-learn pandas


  Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl (11.5 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import cv2import os
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score


In [4]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generate some sample data
def generate_sample_data(n_samples=1000):
    np.random.seed(42)
    
    # Create legitimate features
    X = np.random.randn(n_samples, 5)
    
    # Simple decision rule: if sum of features > 0, class 1, else class 0
    y = (np.sum(X, axis=1) > 0).astype(int)
    
    return X, y

# Function that simulates a vulnerable ML service
class VulnerableMLService:
    def __init__(self):
        # Train the model on legitimate data
        X, y = generate_sample_data()
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.model.fit(self.X_train, self.y_train)
        
        # Check accuracy
        y_pred = self.model.predict(self.X_test)
        self.accuracy = accuracy_score(self.y_test, y_pred)
        print(f"Model trained with accuracy: {self.accuracy:.4f}")
        
        # Define expected input ranges based on training data
        self.min_values = np.min(self.X_train, axis=0)
        self.max_values = np.max(self.X_train, axis=0)
        
    def predict(self, X, validate=True):
        """
        Make predictions with optional input validation
        """
        if validate:
            # Check if inputs are within expected ranges
            for i in range(X.shape[1]):
                if np.any(X[:, i] < self.min_values[i]) or np.any(X[:, i] > self.max_values[i]):
                    return "Error: Input values out of expected range"
        
        # Make prediction
        return self.model.predict(X)

# Demonstrating the attack
if __name__ == "__main__":
    # Create our vulnerable service
    ml_service = VulnerableMLService()
    
    # Create a legitimate test sample
    X_legitimate = np.array([[0.1, 0.2, 0.3, 0.4, 0.5]])
    print("\nLegitimate prediction:")
    print(f"Input: {X_legitimate}")
    print(f"Output: {ml_service.predict(X_legitimate)}")
    
    # Create an adversarial sample with extreme values
    X_adversarial = np.array([[1000, 1000, 1000, 1000, 1000]])
    print("\nAdversarial attack with validation:")
    print(f"Input: {X_adversarial}")
    print(f"Output: {ml_service.predict(X_adversarial)}")
    
    # Demonstrate bypassing validation
    print("\nAdversarial attack without validation:")
    print(f"Input: {X_adversarial}")
    print(f"Output: {ml_service.predict(X_adversarial, validate=False)}")
    
    # Create a more subtle adversarial example that bypasses validation
    # but is designed to trigger a specific prediction
    X_stealthy = np.array([[
        ml_service.max_values[0],
        ml_service.max_values[1],
        ml_service.max_values[2],
        ml_service.max_values[3],
        ml_service.max_values[4]
    ]])
    print("\nStealthy adversarial attack (bypasses validation):")
    print(f"Input: {X_stealthy}")
    print(f"Output: {ml_service.predict(X_stealthy)}")
    
    # Demonstrate a type confusion attack
    try:
        print("\nType confusion attack:")
        bad_input = "string input instead of numpy array"
        print(f"Input: {bad_input}")
        print(f"Output: {ml_service.predict(bad_input)}")
    except Exception as e:
        print(f"Error: {str(e)}")

Model trained with accuracy: 0.8950

Legitimate prediction:
Input: [[0.1 0.2 0.3 0.4 0.5]]
Output: [1]

Adversarial attack with validation:
Input: [[1000 1000 1000 1000 1000]]
Output: Error: Input values out of expected range

Adversarial attack without validation:
Input: [[1000 1000 1000 1000 1000]]
Output: [1]

Stealthy adversarial attack (bypasses validation):
Input: [[3.92623771 3.24309297 3.15205673 3.07888081 3.85273149]]
Output: [1]

Type confusion attack:
Input: string input instead of numpy array
Error: 'str' object has no attribute 'shape'
